In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel , Field
from typing import Literal
from langchain_core.runnables import RunnableParallel,RunnableBranch,RunnableLambda



In [19]:
load_dotenv()

True

In [20]:
model = ChatGoogleGenerativeAI (
    model="gemini-2.5-flash",
    temperature=0,
)

In [21]:
parser = StrOutputParser()

In [22]:
class Feedback(BaseModel):
    sentiment: Literal['positive','Negative'] = Field(description='Give the sentiment of the feedback')

In [23]:
parser2 = PydanticOutputParser(pydantic_object = Feedback)

In [24]:
prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variable = ['feedback'],
    partial_variables = {'format_instruction': parser2.get_format_instructions()}

)

In [25]:
classifier_chain = prompt1 | model | parser2

In [26]:
# result = classifier_chain.invoke({'feedback' : 'This is a terrible smartphone'})

Here we can see that we are getting positive or clearly a single word but we do not have control over it ...
The LLM can also print "The sentiment is Positive/Negative"!!

We need to ensure that the output always remains as Negative/Positive basically a single word defining the sentiment 
So that it works smoothly when used as input to others.

In [27]:
# print(result) ## prints Negative or positive in a json format

In [28]:
prompt2 = PromptTemplate(
    template = 'Write an appropriate response to this positive feedback \n {feedback}',
    input_variables= ['feedback']
)

In [29]:
prompt3 = PromptTemplate(
    template = 'Write an appropriate response to this negative feedback \n {feedback}',
    input_variables= ['feedback']
)

In [30]:
branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'Negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

In [31]:
chain = classifier_chain | branch_chain

In [32]:
print(chain.invoke({'feedback': 'This is a beautiful phone'}))

That's great news! Responding to positive feedback is a fantastic opportunity to reinforce customer loyalty and show appreciation.

Here are a few options, ranging from general to slightly more specific, depending on the context:

**1. General & Appreciative:**

*   "Thank you so much for your kind words! We truly appreciate your positive feedback."
*   "That's wonderful to hear! We're so glad you had a positive experience."
*   "We really appreciate you taking the time to share your feedback. It means a lot to us!"
*   "Thank you! We're delighted to know you're happy."

**2. Enthusiastic & Warm:**

*   "Wow, thank you! We're absolutely thrilled to hear that. Your feedback truly brightens our day!"
*   "That's fantastic! We're so happy we could provide you with a positive experience."
*   "We're over the moon to receive such positive feedback! Thank you for sharing."

**3. Slightly More Formal/Professional:**

*   "Thank you for your valuable feedback. We are pleased to learn that you 

In [34]:
print(chain.invoke({'feedback': 'This is a terrible phone'}))

When responding to negative feedback, the key is to be empathetic, acknowledge the issue, apologize sincerely, and offer a path to resolution. Since I don't have the specific feedback, I'll provide a few templates you can adapt based on the situation.

**Core Principles for Responding to Negative Feedback:**

1.  **Acknowledge and Validate:** Show you've heard them and understand their frustration.
2.  **Apologize Sincerely:** Take responsibility, even if it's just for their negative experience.
3.  **Empathize:** Show you care about their feelings and experience.
4.  **Seek More Information (if needed):** If the feedback is vague, ask for details to understand the root cause.
5.  **Offer a Solution or Path to Resolution:** Don't just apologize; offer to make it right.
6.  **Thank Them:** Even negative feedback is valuable for improvement.
7.  **Move Offline (for public feedback):** If it's on social media, try to take the conversation to a private channel.

---

**Template 1: General 

In [33]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
